# Lakehouse explorationInteractive tour of the medallion layers. Run `make seed && make demo` first so thelocal Delta lake exists.

In [ ]:
import syssys.path.insert(0, "../src")from lakehouse.config import load_env, load_datasetsfrom lakehouse.spark_session import get_sparkcfg = load_env("local")spark = get_spark("exploration", cfg)print("catalog:", cfg.catalog, "| root:", cfg.root_path)

## Dataset contractsEverything the pipeline knows about a source is declarative.

In [ ]:
for name, ds in load_datasets().items():    print(f"{name:>10}: pk={ds.primary_keys} scd={ds.scd} "          f"partition={ds.partition_by} evolution={ds.schema_evolution}")

## Bronze — raw with provenanceNote the `_ingest_*` columns and that everything is still STRING.

In [ ]:
spark.table("bronze.bronze_orders").printSchema()spark.table("bronze.bronze_orders").select(    "order_id", "quantity", "unit_price", "_ingest_date", "_source_file").show(5, truncate=False)

## Silver — typed, deduplicated, enrichedMoney columns are computed once here so every consumer agrees.

In [ ]:
spark.sql('''    SELECT order_id, quantity, unit_price, discount_pct,           gross_amount, discount_amount, net_amount, status    FROM silver.silver_orders    ORDER BY net_amount DESC    LIMIT 5''').show(truncate=False)

## Quarantine — what the quality gate rejected, and why

In [ ]:
spark.sql('''    SELECT _dataset, explode(_dq_failed_rules) AS failed_rule, count(*) AS rows_rejected    FROM ops.quarantine    GROUP BY ALL    ORDER BY rows_rejected DESC''').show(truncate=False)

## SCD2 — customer historyCustomers whose attributes changed have two rows: one closed, one current.

In [ ]:
spark.sql('''    SELECT customer_id, segment, city, effective_from, effective_to, is_current    FROM silver.silver_customers    WHERE customer_id IN (        SELECT customer_id FROM silver.silver_customers        GROUP BY customer_id HAVING count(*) > 1    )    ORDER BY customer_id, effective_from    LIMIT 10''').show(truncate=False)

## Gold — the star schema

In [ ]:
spark.sql('''    SELECT order_date, order_count, net_revenue, gross_profit, margin_pct    FROM gold.agg_daily_sales    ORDER BY net_revenue DESC    LIMIT 10''').show()

## Delta features: time travel and history

In [ ]:
spark.sql("DESCRIBE HISTORY gold.fact_orders").select(    "version", "timestamp", "operation", "operationMetrics").show(5, truncate=80)

In [ ]:
# Read the table as it was at version 0spark.sql("SELECT count(*) AS rows_at_v0 FROM gold.fact_orders VERSION AS OF 0").show()

## Pipeline telemetryThe pipeline's own metrics are Delta tables, queryable like any other.

In [ ]:
spark.sql('''    SELECT job_name, dataset, layer, status, rows_written, round(duration_s, 1) AS secs    FROM ops.pipeline_metrics    ORDER BY started_at DESC    LIMIT 15''').show(truncate=False)

## Schema evolution history

In [ ]:
spark.sql('''    SELECT dataset, layer, version, policy, diff_json    FROM ops.schema_registry    ORDER BY created_at DESC''').show(truncate=100)